# core

> `core` contains the `Unknown` class — a pseudonymisation/anonymisation engine for consistent cross-table key replacement and column masking, with Fernet-encrypted storage of mappings.

In [ ]:
#| default_exp core

I've done some prior exploration here's what I propose. 

**Project: `unknown`** — a pseudonymisation/anonymisation package built with nbdev (fast.ai style). The agreed design:

- **One class** (e.g. `Anonymiser`) with shared state across tables
- **`key_maps`**: `{group_id: {original_val: pseudonym}}` — for consistent cross-table key pseudonymisation
- **`vault`**: stores originals of masked columns so they can be restored
- **Config per table** using key groups like `id1`, `id2`, etc.:
  ```python
  tables = {
      'patients': {'id1': 'mid',       'mask': ['name', 'dob']},
      'visits':   {'id1': 'mid',       'id2': 'visit_code'},
      'doctors':  {'id2': 'doc_id',    'mask': ['name']},
  }
  ```
- **Lookup table approach** (random UUIDs, stored in a dict) — simple and reversible

The nbdev project is already scaffolded, and we're in `00_core.ipynb` which currently just has `def foo(): pass`. 

## Setup & data

In [ ]:
import httpx
from pathlib import Path

base = "https://raw.githubusercontent.com/neo4j-contrib/northwind-neo4j/refs/heads/master/data"
files = ["customers.csv", "orders.csv", "employees.csv"]
dest = Path("../data")
dest.mkdir(exist_ok=True)

for f in files:
    (dest/f).write_bytes(httpx.get(f"{base}/{f}").content)
    print(f"✓ {f}")

In [ ]:
import pandas as pd
for f in files:
    df = pd.read_csv(dest/f,on_bad_lines='skip' )
    print(f"\n{f}: {df.shape}")
    print(df.columns.tolist())
    display(df.head(2))

## Table configuration

In [ ]:
tables = {
    'customers': {
        'id1': 'customerID',
        'mask': ['contactName', 'address', 'phone', 'companyName', 'fax', 'city']
    },
    'orders': {
        'id1': 'customerID',
        'id2': 'employeeID',
        'mask': ['shipName', 'shipAddress']
    },
    'employees': {
        'id2': 'employeeID',
        'mask': ['firstName', 'lastName', 'address', 'homePhone', 'birthDate', 'notes']
    },
}

Okay can we sketch this out? I started with these data structures and the North Wind dataset #Should I start building the Unknown class; We need English UK 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great config — that maps nicely to the design from the start notebook. A few thoughts on the sketch:

**Class name:** `Unknown` (as you suggested) — and we'll use UK English throughout (`anonymise`, `pseudonymise`, etc.)

**Core state:**
- `tables` — the config you've already written
- `key_maps` — `{group: {orig_val: pseudonym}}` e.g. `{'id1': {'ALFKI': 'uuid-xxx', ...}}`
- `vault` — stores original masked column values per table, so you can restore them

**Key methods to start with:**
1. `pseudonymise(name, df)` — takes a table name + DataFrame, pseudonymises keys and masks columns, returns the cleaned DataFrame
2. `restore(name, df)` — reverses the process

Does that match your thinking? If so, go ahead and stub out the class with `__init__` — just setting up those three attributes from the config. I'd suggest starting simple: take `tables` as the only `__init__` parameter.

## Building the `Unknown` class

In [ ]:
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df): pass
    def decode(self, name,df): pass



#Simple start. Let's review.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice clean stub! A few notes:

- **`encode`/`decode`** — good naming choice, cleaner than `pseudonymise`/`restore`
- `(object)` — you can drop that, it's implicit in Python 3, but no harm keeping it
- You might want `#| export` above the class so nbdev exports it to your module

For `encode`, the logic will be roughly:
1. **Pseudonymise keys** — for each `id*` group in `tables[name]`, map original values → UUIDs (reusing existing mappings from `key_maps`)
2. **Vault & mask** — for each column in `mask`, stash the originals in `vault[name]`, then blank/redact them

Want to tackle step 1 first — the key pseudonymisation bit inside `encode`?

## Key pseudonymisation

In [ ]:
for t in tables.keys():
    print("table is->", t)
    print(tables[t])

In [ ]:
ids = {k: v for k, v in tables['orders'].items() if k.startswith('id')}
ids

In [ ]:
customers = pd.read_csv(dest/"customers.csv", on_bad_lines='skip')
customers.sample(2)

In [ ]:
unique_customers = customers['customerID'].unique()
unique_customers
import uuid
u = uuid.uuid4()

OK, I got as far as this. Before I continue, explain briefly:

1. why uuid4 is this a more secure id format
2. i assume i can from each uuid transform this into hex, int or other format? that would be the id itself

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good questions!

**1. Why uuid4?** It's generated from **cryptographically random** bits — there's no timestamp, MAC address, or sequence number baked in. That means you can't reverse-engineer any info about *when* or *where* it was created. With 122 random bits, the chance of a collision is astronomically small (you'd need to generate billions per second for decades). That's what makes it ideal for pseudonymisation — the fake ID reveals nothing about the original.

**2. Formats** — Yes, exactly. A UUID is just a 128-bit number. You can represent it as:
- `str(u)` → `'a3b2c1d4-...'` (the familiar hyphenated hex)
- `u.hex` → `'a3b2c1d4...'` (32 hex chars, no hyphens)
- `u.int` → a big integer
- `u.bytes` → 16 raw bytes

For your key maps, `str(u)` is probably the most readable choice. You could also use `u.hex` if you want something shorter and cleaner.

Ready to build that `{original: uuid}` mapping dict for the customerID column?

In [ ]:
{i:str(uuid.uuid4()) for i in unique_customers[:3]}

OK, let's rewrite this into the class. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — that comprehension is exactly the core logic. Now think about where it fits in `encode`:

1. Extract the id groups (you already know how: `k.startswith('id')`)
2. For each group, check if `self.key_maps[group]` already exists — if so, reuse it; if not, build it from the unique values
3. Replace the column values using the map

A hint for step 2–3: `df[col].map(mapping)` will swap values using a dict.

Have a go at writing the key pseudonymisation part of `encode` — just that bit, not the masking yet.

In [ ]:
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df):
        # extract the ids
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            if group not in self.key_maps: self.key_maps[group] = {}
            uniques = df[col].dropna().unique()
            for u in uniques:
                if u not in self.key_maps[group]: self.key_maps[group][u] = str(uuid.uuid4())
    def decode(self, name,df): pass



I'm not super happy with my code so far, let's review if I can make this more readable

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Your logic is solid — it's just a bit dense. Two quick wins:

1. **`setdefault`** replaces the `if group not in` check:
   ```python
   km = self.key_maps.setdefault(group, {})
   ```
   This returns the existing dict or creates an empty one — one line instead of two.

2. **Inner loop** — once you have `km`, you can tighten it:
   ```python
   for val in df[col].dropna().unique():
       km.setdefault(val, str(uuid.uuid4()))
   ```
   Same pattern — only inserts if the key is missing.

That removes both `if` statements and makes the intent clearer. Give those two changes a try and see how it reads.

I've never used setdefault; could you explain this a bit more? Is this a recent Python addition?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠

Not recent at all — it's been in Python since version 2.0! It's just one of those dict methods that many people overlook.

Here's what it does:

```python
d = {}
d.setdefault('x', 42)   # 'x' not in d, so it sets d['x'] = 42 and returns 42
d.setdefault('x', 99)   # 'x' already in d, so it does nothing and returns 42
```

So it's a **get-or-insert** in one call. It always returns the value — either the existing one or the newly inserted default. That's what makes it perfect for building up dicts of dicts:

```python
km = self.key_maps.setdefault(group, {})
```

After this line, `km` is a reference to the inner dict — whether it already existed or was just created. So any changes you make to `km` are reflected in `self.key_maps[group]` too.

Does that make sense? Try it out in a quick test cell if you want to see it in action.

In [ ]:
d = {}
d.setdefault('x', 42)   # 'x' not in d, so it sets d['x'] = 42 and returns 42
d.setdefault('x', 99)   # 'x' already in d, so it does nothing and returns 42

In [ ]:
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df):
        df =df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            km = self.key_maps.setdefault(group, {}) # will create or retreive; sourcery :) 
            for val in  df[col].dropna().unique(): km.setdefault(val, str(uuid.uuid4()))
            df[col] = df[col].map(km)
        return df 
    def decode(self, name,df): pass




Let's review this code iteration.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking much cleaner! Key improvements:

- **`df.copy()`** — smart addition, avoids mutating the original DataFrame
- **`setdefault`** — reads much more clearly now
- **`df[col].map(km)`** — does the actual swap, nice and concise
- **Returns `df`** — so you get the pseudonymised version back

One thing to be aware of: `map(km)` will turn any value **not** in the dict into `NaN` — including values that were already `NaN`. That's probably fine for ID columns, but worth knowing.

Want to test it on the customers table and see what comes out?

In [ ]:
unk = Unknown(tables)
unk.encode('customers', customers).sample(2)

OK, this seems to work. Let's add pseudonymise and mask methods and call it when encoding. Please review

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good refactor idea — separate concerns. Here's the shape I'd suggest:

**`_pseudonymise(self, name, df)`** — extracts the key-mapping logic you already have in `encode` (the `ids` loop). Returns the modified `df`.

**`_mask(self, name, df)`** — stores the original values of the `mask` columns in `self.vault[name]`, then replaces them with something (e.g. `'***'` or `None`). Returns the modified `df`.

**`encode`** becomes just:
```python
def encode(self, name, df):
    df = df.copy()
    df = self._pseudonymise(name, df)
    df = self._mask(name, df)
    return df
```

I'd use underscore-prefixed names (`_pseudonymise`, `_mask`) since they're internal — the public API is just `encode`/`decode`.

For `_mask`, the key question is: how do you want to store originals in the vault? One natural choice is `self.vault[name] = df[mask_cols].copy()` — stash the whole sub-DataFrame before blanking.

Have a go at writing those two methods!

## Column masking

In [ ]:
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df):
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)
 
    def _pseudonymise_keys(self, name,df):
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            km = self.key_maps.setdefault(group, {})
            for val in  df[col].dropna().unique(): km.setdefault(val, str(uuid.uuid4()))
            df[col] = df[col].map(km)
        return df
    
    def _mask_columns(self, name, df): pass     
    
    
    def decode(self, name,df): pass




In [ ]:
unk = Unknown(tables)

unk._pseudonymise_keys('customers', customers).sample(10)


Let's use a hex truncated key for the mask and store in a dict across all dataframes. This way same names or addresses can be retrieved. Also let's allow nulls here not a problem to decode. I will address these in ids

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking — a shared vault means if the same address appears in both `customers` and `orders`, it gets the same pseudonym. So `vault` becomes a flat lookup like `key_maps`:

- `self.vault = {}` — `{original_value: truncated_hex}` shared across all tables
- NaNs pass through untouched
- Something like `uuid.uuid4().hex[:12]` for the replacement (you pick the length)

The logic in `_mask_columns` would be very similar to `_pseudonymise_keys` — loop over the `mask` columns, build mappings in `self.vault`, then `df[col].map(self.vault)`. The key difference is you'd want to skip NaNs rather than mapping them.

Hint: `df[col].map(self.vault)` already leaves NaN as NaN, so that's handled for free.

Have a go at implementing `_mask_columns`?

In [ ]:
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}

    def encode(self, name, df):
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def _remap(self, df, col, mapping, id_fn):
        for val in df[col].dropna().unique():
            mapping.setdefault(val, id_fn())
        df[col] = df[col].map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def decode(self, name, df): pass

OK, review code for now after latest changes; highlight these.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice iteration! Here's what changed and why it's good:

**`_remap` helper** — You extracted the common pattern (loop uniques → setdefault → map) into one method, with `id_fn` as the generator. Both `_pseudonymise_keys` and `_mask_columns` now use it. That's a solid DRY refactor.

**`assert` on key columns** — Smart. IDs shouldn't be null — catching that early avoids silent bugs downstream. Good call separating this concern from `_mask_columns` which is fine with nulls.

**Shared `self.vault`** — Flat dict across all tables, so `"Maria Anders"` gets the same hex whether it appears in customers or elsewhere. Exactly what you described.

**`hex[:8]`** — 8 hex chars = 32 bits ≈ 4 billion possibilities. Should be fine for this dataset, but worth considering if you'd want longer for production use.

One thing to watch: `_remap` is called from both methods, but `_pseudonymise_keys` does the `df.copy()` while `_mask_columns` doesn't — that's fine since `_pseudonymise_keys` always runs first in `encode`, but it's an implicit dependency. Something to keep in mind.

Want to test it end-to-end now?

In [ ]:
unk = Unknown(tables)
for name in tables:
    df = pd.read_csv(dest/f"{name}.csv", on_bad_lines='skip')
    print(f"\n--- {name} ---")
    display(unk.encode(name, df).head(3))

In [ ]:
unk.key_maps

In [ ]:
unk.vault

Let's work on decoding this now, the inverse property. Requirement: I also would like something to store the encoded values. Now let's just not assume that these would be available within the class because likely this could be used at the end of a pipeline 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking — in a real pipeline, encoding and decoding would likely happen in separate steps (maybe even separate machines). So you need two things:

1. **Persist the mappings** — `key_maps` and `vault` need to be saveable/loadable (JSON is natural since they're just nested dicts of strings)
2. **`decode`** — invert the mappings (`{pseudonym: original}`) and `.map()` back

A couple of design questions for you:

- For storage, are you happy with a simple `save(path)` / `load(path)` pair using JSON? Or do you want something fancier?
- Should `decode` work from the in-memory state (if available) **and** from a loaded file? Or only from file?

For a start, simple save and restore is fine. But given this is sensitive data, I would like to discuss some encryption at rest.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good instinct — those mappings are essentially the keys to de-anonymise everything, so they're the most sensitive artefact in the whole pipeline.

For encryption at rest, the simplest solid option is **Fernet** from the `cryptography` library:
- Symmetric encryption (one key encrypts and decrypts)
- AES-128-CBC under the hood, with HMAC authentication
- Password → key derivation via PBKDF2 is built in

The flow would be:
1. **Save**: JSON serialize → encrypt with Fernet → write bytes to file
2. **Load**: read bytes → decrypt with Fernet → JSON deserialize

The main design choice: do you want a **password-based** key (user supplies a passphrase each time) or a **generated key** stored separately (e.g. in an env var or key file)?

## Encryption at rest

In [ ]:
from cryptography.fernet import Fernet

key = Fernet.generate_key()    # bytes, save this somewhere safe
f = Fernet(key)

encrypted = f.encrypt(b"hello")  # bytes in → bytes out
f.decrypt(encrypted)             # b"hello"

encrypted

Assume I use Fernet. Let's rethink the class to have this option to generate the key and store it first. I want it outside its state like a static method maybe

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call — keeping the key outside instance state means the `Unknown` object itself is never holding the secret. Clean separation.

The shape would be:

- **`Unknown.generate_key()`** — `@staticmethod`, returns a Fernet key (bytes). User stores it however they like (env var, file, vault)
- **`save(path, key)`** — serialise `key_maps` + `vault` to JSON, encrypt with the provided key, write to file
- **`load(path, key)`** — read file, decrypt, deserialise, return a new `Unknown` instance (or populate `key_maps`/`vault`)

So the key is always **passed in** at save/load time, never stored on `self`. The user is responsible for keeping it safe.

Want to start by adding the `generate_key` staticmethod?

In [ ]:
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}

    def encode(self, name, df):
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def _remap(self, df, col, mapping, id_fn):
        for val in df[col].dropna().unique():
            mapping.setdefault(str(val), id_fn()) # add a new key if not found to val (id_fn() return)
        df[col] = df[col].astype(str).map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        df = df.copy()
        
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            assert pd.api.types.is_string_dtype(df[col]), f"Column {col} must be string type, got {df[col].dtype}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def decode(self, name, df): pass

    @staticmethod
    def generate_key(path=None):
        from cryptography.fernet import Fernet
        key = Fernet.generate_key()
        if path:
            Path(path).write_bytes(key)
        return key
    
    @staticmethod
    def load_key(path=None, env_var=None):
        if path: return Path(path).read_bytes()
        if env_var:
            import os
            return os.environ[env_var].encode()
        raise ValueError("Provide either path or env_var")





In [ ]:
# key = Unknown.generate_key()
# key.decode()

let's discuss where to include the fernet key load. I would argue at init. the user should have one set prior to using this 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a reasonable argument — it ensures encryption is always available. But consider the trade-off:

**Key at `__init__`** — enforces that encryption is set up from the start. `save()`/`load()` can just use `self._key` internally. Clean API.

**Key at `save()`/`load()` only** — someone who just wants to `encode` a DataFrame in memory (no persistence) doesn't need to deal with keys at all. Lower barrier for simple use cases.

A middle ground: make it **optional** at init, but **required** for save/load. Something like:

```python
def __init__(self, tables, key=None):
```

If `key` is set, `save()` just works. If not, `save()` raises a clear error. That way casual usage stays simple, but the pipeline path is solid.

What's your preference?

In [ ]:
from fastcore.basics import patch
import json

@patch
def save(self: Unknown, path, key):
    """Encrypt and save key_maps + vault to file."""
    from cryptography.fernet import Fernet
    data = json.dumps({'key_maps': self.key_maps, 'vault': self.vault})
    encrypted = Fernet(key).encrypt(data.encode())
    Path(path).write_bytes(encrypted)

@patch(cls_method=True)
def load(cls: Unknown, path, key, tables):
    """Load encrypted mappings and return a new Unknown instance."""
    from cryptography.fernet import Fernet
    encrypted = Path(path).read_bytes()
    data = json.loads(Fernet(key).decrypt(encrypted))
    unk = cls(tables)
    unk.key_maps, unk.vault = data['key_maps'], data['vault']
    return unk

In [ ]:
bad_key = Unknown.generate_key()
try:
    Unknown.load(dest/"mappings.enc", bad_key, tables)
    assert False, "Should have raised"
except Exception as e:
    print(f"✓ Correctly rejected: {type(e).__name__}")


In [ ]:
@patch
def decode(self: Unknown, name, df):
    """Reverse pseudonymisation and masking for key columns and masked columns"""
    df = df.copy()
    tbl = self.tables[name]
    # Inverse key_maps for each id-group
    ids = {k: v for k, v in tbl.items() if k.startswith('id')}
    for group, col in ids.items():
        km = self.key_maps[group]
        inv_km = {v: k for k, v in km.items()}
        df[col] = df[col].map(inv_km)
    # Masked columns
    inv_vault = {v: k for k, v in self.vault.items()}
    for col in tbl.get('mask', []):
        df[col] = df[col].map(inv_vault)
    return df

In [ ]:
import os

# 1. Generate key and store in env var
key = Unknown.generate_key()
os.environ['UNKNOWN_KEY'] = key.decode()
loaded_key = Unknown.load_key(env_var='UNKNOWN_KEY')
assert loaded_key == key
print("✓ Key round-tripped via env var")

# 2. Encode all tables
unk = Unknown(tables)
originals, encoded = {}, {}
for name in tables:
    df = pd.read_csv(dest/f"{name}.csv", on_bad_lines='skip')
    for k, v in tables[name].items():
        if k.startswith('id'): df[v] = df[v].astype(str)
    originals[name] = df
    encoded[name] = unk.encode(name, originals[name])
    print(f"✓ Encoded {name}")

# 3. Save, load with env key, decode and compare
unk.save(dest/"mappings.enc", loaded_key)
unk2 = Unknown.load(dest/"mappings.enc", loaded_key, tables)
for name in tables:
    decoded = unk2.decode(name, encoded[name])
    pd.testing.assert_frame_equal(decoded, originals[name])
    print(f"✓ Decoded {name} matches original")

## Final class & round-trip test

In [ ]:
#| export
#| export
import uuid, json
from pathlib import Path
from cryptography.fernet import Fernet
import pandas as pd


class Unknown:
    """Pseudonymise and mask DataFrames consistently across related tables."""
    def __init__(self, tables):
        self.tables,self.vault,self.key_maps = tables,{},{}

    def encode(self, name, df):
        """Pseudonymise keys and mask columns for table `name`, returning new DataFrame."""
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def decode(self, name, df):
        """Reverse `encode` — restore original keys and masked values."""
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            inv = {v: k for k, v in self.key_maps[group].items()}
            df[col] = df[col].map(inv)
        inv_vault = {v: k for k, v in self.vault.items()}
        for col in tbl.get('mask', []):
            df[col] = df[col].map(inv_vault)
        return df

    def _remap(self, df, col, mapping, id_fn):
        """Replace `col` values via `mapping`, adding new entries with `id_fn`."""
        for val in df[col].dropna().unique():
            mapping.setdefault(str(val), id_fn())
        df[col] = df[col].astype(str).map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        """Replace id columns with consistent UUIDs via `key_maps`."""
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            assert pd.api.types.is_string_dtype(df[col]), f"Column {col} must be string type, got {df[col].dtype}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        """Replace mask columns with truncated hex tokens via shared `vault`."""
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def save(self, path, key):
        """Encrypt and save `key_maps` + `vault` to `path` using Fernet `key`."""
        data = json.dumps({'key_maps': self.key_maps, 'vault': self.vault})
        Path(path).write_bytes(Fernet(key).encrypt(data.encode()))

    @classmethod
    def load(cls, path, key, tables):
        """Load encrypted mappings from `path` and return a new `Unknown` instance."""
        data = json.loads(Fernet(key).decrypt(Path(path).read_bytes()))
        unk = cls(tables)
        unk.key_maps, unk.vault = data['key_maps'], data['vault']
        return unk

    @staticmethod
    def generate_key(path=None):
        """Generate a Fernet key, optionally saving to `path`."""
        key = Fernet.generate_key()
        if path: Path(path).write_bytes(key)
        return key

    @staticmethod
    def load_key(path=None, env_var=None):
        """Load a Fernet key from `path` or `env_var`."""
        if path: return Path(path).read_bytes()
        if env_var:
            import os
            return os.environ[env_var].encode()
        raise ValueError("Provide either path or env_var")

now guide me on how to publish this applciation.. i dont think i currently have a remote

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's check your current setup first. Can you run:

```bash
!git status && echo "---" && git remote -v
```

That'll tell us where you're at with git and confirm there's no remote. From there we can set up a GitHub repo and get nbdev publishing going.

In [ ]:
!git status && echo "---" && git remote -v

yep not remote

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

OK, let's do this step by step. First — do you have a GitHub token available in your environment? That would let us create the repo right from here using `ghapi`. Run:

```bash
!echo $GITHUB_TOKEN | head -c 10
```

If that's empty, we'll create the repo on GitHub manually instead.

In [ ]:
!echo $GITHUB_TOKEN | head -c 10

no i will do it manually. thsi is the ssh for cloning its an empty repo

`git@github.com:giordafrancis/unknown.git`

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great, let's add the remote and push:

```bash
!git remote add origin git@github.com:giordafrancis/unknown.git
```

Then commit your current changes and push:

```bash
!git add -A && git commit -m "Initial Unknown class with encode/decode and Fernet encryption"
!git push -u origin master
```

After that's up, we can look at setting up nbdev publishing (docs, PyPI, etc.) if you like.🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great. Since you already have a local repo with commits, just add the remote and push:

```bash
!git remote add origin git@github.com:giordafrancis/unknown.git
!git push -u origin master
```

Once that's pushed, we can look at getting `nbdev` to build your docs and module. Let me know how it goes!

In [ ]:
!git remote add origin git@github.com:giordafrancis/unknown.git

In [ ]:
!git add -A && git commit -m "Initial Unknown class with encode/decode and Fernet encryption"
!git push -u origin master

did we just pushed the key also????

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Don't panic — the **Fernet key** was NOT pushed. It was only stored in `os.environ['UNKNOWN_KEY']` in memory.

What **was** pushed is `data/mappings.enc` — the encrypted mappings file. Without the key it's unreadable, so it's safe. But you probably don't want it in the repo at all.

Let's fix that:

```bash
!echo "data/" >> .gitignore
!git rm -r --cached data/
!git add .gitignore && git commit -m "Remove data dir and add to .gitignore"
!git push
```

This removes it from tracking while keeping the local files. Worth checking your `.gitignore` already has sensible entries too — want to take a look at it first?

In [ ]:
!echo "data/" >> .gitignore
!git rm -r --cached data/
!git add .gitignore && git commit -m "Remove data dir and add to .gitignore"
!git push

Then NBDev package, should I export it? Do I need to pip install locally? Where are the steps? I assume I have to pip install it locally so I can import in index 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Right, the nbdev workflow is:

1. **`nbdev_export`** — reads cells marked `#| export` and generates `.py` modules from your notebooks
2. **`pip install -e .`** — installs the package locally in editable mode, so `import unknown` works everywhere including `index.ipynb`

Try running those two in order:

```bash
!nbdev_export
!pip install -e ..
```

(Note `..` because you're in the `nbs/` folder and `setup.py`/`settings.ini` are in the project root.)

After that, confirm it works:

```python
from unknown.core import Unknown
```

In [ ]:
!nbdev_export
!pip install -e ..

In [ ]:
!nbdev-export

It's not an underscore by an iPhone. This might be a newer version 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're right — newer nbdev uses hyphens: `nbdev-export` not `nbdev_export`. Looks like it ran without errors!

Now try the import:

```python
from unknown.core import Unknown
```

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from unknown.core import Unknown